# ATLAS Hydro stations preprocessing tutorial

This notebook prepares observed river discharge station data for the ATLAS hydro workflow.

The workflow starts from the raw Ecuador hydrological station dataset and produces a clean station climatology file that can be used later to integrate station information with basin statistics and maps.

The notebook is written step by step so that it can be reused by users with limited Python experience.


## 1. What this notebook does

The notebook performs four main operations:

1. reads the station metadata from the station shapefile;
2. reads the observed river discharge time series from the Excel file;
3. cleans the station time series by removing stations with too many missing values, negative values and statistical outliers;
4. computes monthly climatological river discharge values for each station and saves the final table as CSV.

The final output is one row per station and month.


## 2. Import Python libraries

Run this cell first. If one of these libraries is missing, install it in the Python environment before continuing.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd


## 3. Define input and output paths

Edit only this section when applying the workflow to another country or another folder structure.

For Ecuador, the expected folder structure is:

```text
../data/stations/ecuador/
├── shapes_hydro/
│   └── ESTACIONES_HIDROLOGICAS.shp
└── CAUDALES.xlsx
```

The processed output will be saved in:

```text
../data/stations/ecuador/
```


In [2]:
COUNTRY_FOLDER = "ecuador"
COUNTRY_NAME = "Ecuador"

BASE_OUTPUT_DIR = Path("../data/stations") / COUNTRY_FOLDER

STATION_SHAPEFILE = BASE_INPUT_DIR / "shapes_hydro" / "ESTACIONES_HIDROLOGICAS.shp"
STATION_EXCEL_FILE = BASE_INPUT_DIR / "CAUDALES.xlsx"
STATION_EXCEL_SHEET = "Caudal_flags_Finales"

OUTPUT_FILE = BASE_OUTPUT_DIR / f"allstats_river_discharge_{COUNTRY_FOLDER}.csv"

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Country:", COUNTRY_NAME)
print("Station shapefile:", STATION_SHAPEFILE)
print("Station Excel file:", STATION_EXCEL_FILE)
print("Output file:", OUTPUT_FILE)


Country: Ecuador
Station shapefile: /mnt/DATA/PROGETTI/27_WMO_ATLAS/stations/ecuador/shapes_hydro/ESTACIONES_HIDROLOGICAS.shp
Station Excel file: /mnt/DATA/PROGETTI/27_WMO_ATLAS/stations/ecuador/CAUDALES.xlsx
Output file: ../data/stations/ecuador/allstats_river_discharge_ecuador.csv


## 4. Read station metadata

The shapefile contains the station code, name, coordinates and altitude. These fields are used later to attach metadata to the monthly climatology.


In [3]:
if not STATION_SHAPEFILE.exists():
    raise FileNotFoundError(f"Station shapefile not found: {STATION_SHAPEFILE}")

stations_info = gpd.read_file(STATION_SHAPEFILE)

print("Number of stations in shapefile:", len(stations_info))
print("Available columns:")
print(list(stations_info.columns))

stations_info.head()


Number of stations in shapefile: 25
Available columns:
['CODIGO_N', 'NOM_EST', 'TIPO', 'ALTURA', 'LON-G', 'LAT-G', 'LON-GGMMSS', 'LAT-GGMMSS', 'PROVINCIA', 'CUENCA', 'EST_FUN', 'INS_PRO', 'NOM_INTER', 'COD_INTER', 'INS_OPR', 'AREADR', 'FEC_LEV', 'geometry']


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


,CODIGO_N,NOM_EST,TIPO,ALTURA,LON-G,LAT-G,LON-GGMMSS,LAT-GGMMSS,PROVINCIA,CUENCA,EST_FUN,INS_PRO,NOM_INTER,COD_INTER,INS_OPR,AREADR,FEC_LEV,geometry
0,H0011,MIRA EN LITA,LG,475,-78.46361,0.84917,"78G 27' 49"" W","0G 50' 57"" N",8,20,1,84,0,0,84,4960,NaN,POINT (-78.46361 0.84917)
1,H0016,APAQUI AJ CHOTA,LG,1715,-77.93417,0.44639,"77G 56' 3"" W","0G 26' 47"" N",4,20,1,84,0,0,84,703,NaN,POINT (-77.93417 0.44639)
2,H0091,GRANDE AJ JATIVA,LG,3120,-77.84611,0.80417,"77G 50' 46"" W","0G 48' 15"" N",4,30,1,84,0,0,84,54,NaN,POINT (-77.84611 0.80417)
3,H0159,SAN PEDRO EN MACHACHI,LG,2680,-78.54500,-0.46194,"78G 32' 42"" W","0G 27' 43"" S",17,80,1,84,0,0,84,363,NaN,POINT (-78.54500 -0.46194)
4,H0166,TOACHI EN LAS PAMPAS,LG,1110,-78.98194,-0.43667,"78G 58' 55"" W","0G 26' 12"" S",5,80,1,84,0,0,84,1051,NaN,POINT (-78.98194 -0.43667)


## 5. Read observed river discharge data

The Excel file is expected to contain one date column named `fechas` and one column for each station. Station columns are identified because their names start with `H`.


In [4]:
if not STATION_EXCEL_FILE.exists():
    raise FileNotFoundError(f"Station Excel file not found: {STATION_EXCEL_FILE}")

raw_data = pd.read_excel(STATION_EXCEL_FILE, sheet_name=STATION_EXCEL_SHEET)

print("Data shape:", raw_data.shape)
print("Available columns:")
print(list(raw_data.columns))

raw_data.head()


Data shape: (11322, 52)
Available columns:
['fechas', 'mes', 'H0011', 'origen', 'H0016', 'origen.1', 'H0091', 'origen.2', 'H0159', 'origen.3', 'H0166', 'origen.4', 'H0170', 'origen.5', 'H0326', 'origen.6', 'H0331', 'origen.7', 'H0333', 'origen.8', 'H0334', 'origen.9', 'H0337', 'origen.10', 'H0338', 'origen.11', 'H0340', 'origen.12', 'H0347', 'origen.13', 'H0385', 'origen.14', 'H0466', 'origen.15', 'H0471', 'origen.16', 'H0530', 'origen.17', 'H0591', 'origen.18', 'H0616', 'origen.19', 'H0721', 'origen.20', 'H0792', 'origen.21', 'H0793', 'origen.22', 'H0817', 'origen.23', 'H0889', 'origen.24']


,fechas,mes,H0011,origen,H0016,origen.1,H0091,origen.2,H0159,origen.3,...,H0721,origen.20,H0792,origen.21,H0793,origen.22,H0817,origen.23,H0889,origen.24
0,1985-01-01,1,284.041956,GR6J,6.387,H,0.888,H,5.001,H,...,385.700012,H,9.435,H,0.466,H,NaN,H,75.388000,PDM
1,1985-01-02,1,267.988123,GR6J,5.205,H,0.888,H,4.770,H,...,282.799988,H,9.435,H,0.290,H,NaN,H,51.380001,H
2,1985-01-03,1,254.669535,GR6J,6.639,H,0.888,H,4.328,H,...,193.399994,H,9.165,H,0.259,H,NaN,H,48.669998,H
3,1985-01-04,1,244.075427,GR6J,5.432,H,0.954,H,4.328,H,...,155.500000,H,9.435,H,0.181,H,NaN,H,46.060001,H
4,1985-01-05,1,235.703224,GR6J,5.205,H,0.954,H,4.118,H,...,172.399994,H,9.435,H,0.233,H,NaN,H,41.099998,H


## 6. Standardise the date column and identify station columns

The original date column is renamed from `fechas` to `time`. This makes the dataset easier to combine with other ATLAS hydro outputs.


In [5]:
data = raw_data.copy()

if "fechas" not in data.columns:
    raise KeyError("The expected date column 'fechas' was not found in the Excel file.")

data["fechas"] = pd.to_datetime(data["fechas"], errors="coerce")
data = data.rename(columns={"fechas": "time"})

data = data.dropna(subset=["time"])

station_columns = [col for col in data.columns if str(col).startswith("H")]

if len(station_columns) == 0:
    raise ValueError("No station columns were found. Expected columns starting with 'H'.")

print("Number of station columns found:", len(station_columns))
print(station_columns[:10])


Number of station columns found: 25
['H0011', 'H0016', 'H0091', 'H0159', 'H0166', 'H0170', 'H0326', 'H0331', 'H0333', 'H0334']


## 7. Remove stations with too many missing values

Stations with at least 30% missing values are removed. You can change the threshold if needed.


In [6]:
MAX_MISSING_FRACTION = 0.30

missing_fraction = data[station_columns].isna().mean()
stations_to_drop = missing_fraction[missing_fraction >= MAX_MISSING_FRACTION].index.tolist()

data = data.drop(columns=stations_to_drop)
station_columns = [col for col in data.columns if str(col).startswith("H")]

print("Stations removed because of missing values:", len(stations_to_drop))
print("Stations retained:", len(station_columns))


Stations removed because of missing values: 1
Stations retained: 24


## 8. Remove invalid values and outliers

Negative river discharge values are not physically meaningful, so they are converted to missing values.

Outliers are removed using the interquartile range method for each station separately.


In [7]:
# Convert negative values to NaN.
data[station_columns] = data[station_columns].mask(data[station_columns] < 0)

# Interquartile range outlier filtering.
q1 = data[station_columns].quantile(0.25)
q3 = data[station_columns].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

data[station_columns] = data[station_columns].mask(
    (data[station_columns] < lower_bound) | (data[station_columns] > upper_bound)
)

print("Cleaning completed.")
data[["time"] + station_columns[:5]].head()


Cleaning completed.


,time,H0011,H0016,H0091,H0159,H0166
0,1985-01-01,284.041956,6.387,0.888,5.001,37.669998
1,1985-01-02,267.988123,5.205,0.888,4.770,31.110001
2,1985-01-03,254.669535,6.639,0.888,4.328,25.400000
3,1985-01-04,244.075427,5.432,0.954,4.328,25.990000
4,1985-01-05,235.703224,5.205,0.954,4.118,27.209999


## 9. Convert data from wide to long format

The raw data has one column per station. The ATLAS workflow uses a long table with the columns `time`, `id` and `river_discharge`.


In [8]:
data_long = data.melt(
    id_vars="time",
    value_vars=station_columns,
    var_name="id",
    value_name="river_discharge"
)

data_long = data_long.sort_values(["id", "time"]).reset_index(drop=True)

print("Long table shape:", data_long.shape)
data_long.head()


Long table shape: (271728, 3)


,time,id,river_discharge
0,1985-01-01,H0011,284.041956
1,1985-01-02,H0011,267.988123
2,1985-01-03,H0011,254.669535
3,1985-01-04,H0011,244.075427
4,1985-01-05,H0011,235.703224


## 10. Compute monthly climatology

For each station, the notebook computes the average river discharge for each calendar month.


In [9]:
monthly_climatology = (
    data_long
    .assign(month=data_long["time"].dt.month)
    .groupby(["id", "month"], as_index=False)["river_discharge"]
    .mean()
)

print("Monthly climatology shape:", monthly_climatology.shape)
monthly_climatology.head()


Monthly climatology shape: (288, 3)


,id,month,river_discharge
0,H0011,1,175.314783
1,H0011,2,167.546853
2,H0011,3,181.716048
3,H0011,4,193.501874
4,H0011,5,188.486109


## 11. Prepare station metadata

The following fields are extracted from the station shapefile and renamed using standard names.

If a future country uses different shapefile column names, edit only the dictionary in this cell.


In [10]:
METADATA_COLUMNS = {
    "CODIGO_N": "id",
    "NOM_EST": "name",
    "LAT-G": "latitude",
    "LON-G": "longitude",
    "ALTURA": "altitude",
}

missing_metadata_columns = [col for col in METADATA_COLUMNS if col not in stations_info.columns]
if missing_metadata_columns:
    raise KeyError(
        "The following expected metadata columns were not found in the shapefile: "
        + ", ".join(missing_metadata_columns)
    )

station_metadata = stations_info[list(METADATA_COLUMNS.keys())].copy()
station_metadata = station_metadata.rename(columns=METADATA_COLUMNS)
station_metadata["id"] = station_metadata["id"].astype(str)

monthly_climatology["id"] = monthly_climatology["id"].astype(str)

station_metadata.head()


,id,name,latitude,longitude,altitude
0,H0011,MIRA EN LITA,0.84917,-78.46361,475
1,H0016,APAQUI AJ CHOTA,0.44639,-77.93417,1715
2,H0091,GRANDE AJ JATIVA,0.80417,-77.84611,3120
3,H0159,SAN PEDRO EN MACHACHI,-0.46194,-78.54500,2680
4,H0166,TOACHI EN LAS PAMPAS,-0.43667,-78.98194,1110


## 12. Merge climatology and metadata

The final dataset contains the monthly river discharge climatology plus station name, latitude, longitude and altitude.


In [11]:
stations_final = monthly_climatology.merge(station_metadata, on="id", how="left")

missing_metadata = stations_final[stations_final["latitude"].isna() | stations_final["longitude"].isna()]["id"].unique()

print("Final table shape:", stations_final.shape)
print("Stations without metadata:", len(missing_metadata))
if len(missing_metadata) > 0:
    print(missing_metadata[:20])

stations_final.head()


Final table shape: (288, 7)
Stations without metadata: 0


,id,month,river_discharge,name,latitude,longitude,altitude
0,H0011,1,175.314783,MIRA EN LITA,0.84917,-78.46361,475
1,H0011,2,167.546853,MIRA EN LITA,0.84917,-78.46361,475
2,H0011,3,181.716048,MIRA EN LITA,0.84917,-78.46361,475
3,H0011,4,193.501874,MIRA EN LITA,0.84917,-78.46361,475
4,H0011,5,188.486109,MIRA EN LITA,0.84917,-78.46361,475


## 13. Save the processed station file

The output CSV is used by the following station integration and mapping notebook.


In [12]:
stations_final.to_csv(OUTPUT_FILE, index=False)

print("Saved file:", OUTPUT_FILE)
print("Number of rows saved:", len(stations_final))


Saved file: ../data/stations/ecuador/allstats_river_discharge_ecuador.csv
Number of rows saved: 288


## 14. Quick quality check

This optional check shows the number of months available for each station in the final output.


In [13]:
quality_check = (
    stations_final
    .groupby("id")
    .agg(
        months_available=("month", "nunique"),
        mean_river_discharge=("river_discharge", "mean"),
        station_name=("name", "first")
    )
    .reset_index()
    .sort_values("id")
)

quality_check.head(20)


,id,months_available,mean_river_discharge,station_name
0,H0011,12,149.496479,MIRA EN LITA
1,H0016,12,8.425256,APAQUI AJ CHOTA
2,H0091,12,1.580857,GRANDE AJ JATIVA
3,H0159,12,4.216475,SAN PEDRO EN MACHACHI
4,H0166,12,17.583608,TOACHI EN LAS PAMPAS
5,H0170,12,301.091875,GUAYLLABAMBA AJ BLANCO
6,H0326,12,100.273145,BABA DJ TOACHI-DCP
7,H0331,12,4.154137,CHIMBO EN SAN LORENZO
8,H0333,12,0.731272,SAN LORENZO EN SAN LORENZO
9,H0334,12,1.058206,DE CHIMA AJ LAS PESQ.(SICOTO AJ)
